# Taking the state away from a working controller

A tractor towing passive trailers has to follow a path, and the controller needs the pose
of **every** segment to do it. In the field it gets far less: the angles at the joints, the
pose of one segment, and wheels that slip.

A re-implementation of

> N. Deniz, F. Jorquera, M. Torres-Torriti and F. Auat Cheein, *Model predictive
> path-following controller for Generalised N-Trailer vehicles with noisy sensors and
> disturbances*, Control Engineering Practice 142:105747, 2024.
> [doi:10.1016/j.conengprac.2023.105747](https://doi.org/10.1016/j.conengprac.2023.105747)

This is not the code behind the paper and its numbers are not the paper's numbers — see
[`NOTES.md`](https://github.com/nahueldeniz/nmhec/blob/main/NOTES.md). What it reproduces is
the *mechanism*, and the fastest way to understand a mechanism is to break it.

In [ ]:
try:
    import nmhec  # noqa: F401
except ImportError:
    !pip install -q git+https://github.com/nahueldeniz/nmhec.git
    import nmhec  # noqa: F401

import matplotlib.pyplot as plt
import numpy as np

from nmhec import (ClosedLoop, EstimatorConfig, Ground, PFAConfig, SensorConfig,
                   VehicleGeometry)
from nmhec.path import lemniscate
from nmhec.viz import compare_bars, plot_diagnostics, plot_run

plt.rcParams['figure.dpi'] = 110
PATH = lemniscate(scale=0.6).sample(600)[:, :2]
GEOMETRY, TS = VehicleGeometry(n_trailers=2), 0.1

def run(estimator='nmhe', rho=1.0, n_estimation=10, d_tgt=0.3, measured_segment=0,
        beta_max=np.pi / 2 - np.deg2rad(10), n_steps=300, seed=0):
    loop = ClosedLoop(
        PATH, GEOMETRY, ts=TS, estimator=estimator,
        sensors=SensorConfig(pose_segments=(measured_segment,)),
        ground=Ground(rho_omega=rho, rho_v=rho),
        est_cfg=EstimatorConfig(n_estimation=n_estimation, beta_max=beta_max),
        pfa_cfg=PFAConfig(d_tgt=d_tgt, d_ntgt=d_tgt), seed=seed)
    return loop, loop.run(n_steps)

## 1. It works, and the constraint is doing the work

The vehicle tracks the path at about 0.23 m/s — close to the 0.28 m/s of the paper's field
trials, and slow for the same reason: the controller trades speed for staying on the path.
Watch the bottom panel. The joint angles sit *on* their bound, not near it. The constraint is
not decoration; it is the only thing between this and a jackknife.

In [ ]:
loop, res = run()
plot_run(res, PATH, GEOMETRY, loop.model, title='NMHE, firm ground')
plt.show()
plot_diagnostics(res, TS, loop.ctrl_cfg.beta_max)
plt.show()
res.summary(TS)

### Break it (1): loosen the bound

The paper constrains the joint angles to $\pm(\pi/2-\delta_\beta)$ with
$\delta_\beta = 10\pi/180$, i.e. $\pm 1.396$ rad. Give the chain a little more room and it
folds — and once the true angles leave the range the estimator is allowed to report, the
estimate saturates and the controller is flying blind.

In [ ]:
for bm in (np.pi / 2 - np.deg2rad(10), 1.745, 2.2):
    _, r = run(beta_max=bm, n_steps=200)
    s = r.summary(TS)
    folded = 'JACKKNIFE' if s['beta_max'] > bm + 0.05 else 'ok'
    print(f"bound {bm:.3f} rad -> max |beta| {s['beta_max']:.2f}, "
          f"estimation error {s['estimation_mean']:.3f} m   {folded}")

## 2. Break it (2): turn the ground to mud

`rho = 0.5` means the wheels deliver half the commanded velocity. Nothing downstream is
told. Run each estimator and watch the dashed line — the controller's belief about where the
last trailer is.

In [ ]:
runs = {}
for kind in ('oracle', 'ekf', 'nmhe'):
    loop, res = run(estimator=kind, rho=0.5)
    runs[kind] = (loop, res)
    plot_run(res, PATH, GEOMETRY, loop.model, title=f'{kind}, rho = 0.5')
    plt.show()

summaries = {k: r.summary(TS) for k, (_, r) in runs.items()}
compare_bars(summaries, labels={'estimation_mean': 'estimation error',
                                'infeasibility_mean': 'infeasibility',
                                'off_track_mean': 'off-track error'})
plt.show()

### Why the filter loses, and it is not noise

The paper's state vector $q$ carries every segment's pose as a state of its own, even though
the chain determines them from the tractor's pose and the joint angles. In the moving horizon
estimator those relations are **equality constraints**, so every iterate is a configuration
the vehicle could actually hold. A Kalman filter has no such mechanism: its correction pulls
the whole state towards the measurement, and nothing in it knows the last trailer has to stay
on its hitch.

`feasibility_error` is the distance between the filter's last-trailer pose and the one the
chain implies. Compare it with the estimation error — they are the same number. The filter's
error *is* its infeasibility.

In [ ]:
for kind in ('ekf', 'nmhe'):
    loop, res = runs[kind]
    s = res.summary(TS)
    print(f"{kind:>5}: estimation error {s['estimation_mean']:.3f} m, "
          f"infeasibility {s['infeasibility_mean']:.3f} m")
    plot_diagnostics(res, TS, loop.ctrl_cfg.beta_max, title=f'{kind}, rho = 0.5')
    plt.show()

## 3. Break it (3): starve the estimator

Shorten the window until the estimate stops being worth having. The last column is the cost
in units of the sampling period: above 1 and it does not run in real time. This is the trade
the paper reports — $N_e = 15$ at $T_s = 0.1$ s overran the period, and shortening to 7
brought it back inside.

In [ ]:
print(f"{'N_e':>4} {'est err [m]':>12} {'off-track [m]':>14} {'tau_e / Ts':>11}")
for ne in (2, 5, 10, 20, 30):
    _, res = run(rho=0.9, n_estimation=ne, n_steps=150)
    s = res.summary(TS)
    print(f'{ne:>4} {s["estimation_mean"]:>12.3f} {s["off_track_mean"]:>14.3f} '
          f'{s["rtf_estimator"]:>11.3f}')

## 4. Break the premise: measure a different segment

Eq. (3) asks for the joint angles plus the pose of *at least one* segment — it does not say
which. The simulation study measures the tractor; the real G2T carries its RTK GPS on the
axle of the **last trailer**. Swap it and see what the estimator has to do to propagate that
measurement back up the chain.

In [ ]:
for seg, name in ((0, 'tractor'), (2, 'last trailer')):
    _, res = run(rho=0.9, measured_segment=seg, n_steps=200)
    s = res.summary(TS)
    print(f'measuring the {name:<13} estimation error {s["estimation_mean"]:.3f} m   '
          f'off-track {s["off_track_mean"]:.3f} m')

## 5. The one that catches people out

`D_tgt` decides when the path-following algorithm advances its target. Make it large and the
vehicle cuts corners; the path error is then dominated by that geometry and the estimator
stops mattering. Worth knowing which of your numbers is measuring the thing you think it is.

In [ ]:
print(f"{'D_tgt [m]':>10} {'nmhe off-track':>15} {'ekf off-track':>14}")
for d in (0.2, 0.3, 0.65, 1.0):
    _, a = run(estimator='nmhe', rho=0.5, d_tgt=d, n_steps=150)
    _, b = run(estimator='ekf', rho=0.5, d_tgt=d, n_steps=150)
    print(f'{d:>10.2f} {a.summary(TS)["off_track_mean"]:>15.3f} '
          f'{b.summary(TS)["off_track_mean"]:>14.3f}')

---

## What to take away

On firm ground with Gaussian noise the Kalman filter is the better tool, and cheaper by an
order of magnitude — the comparison at `rho = 1` is not close. It loses when the disturbance
stops being small and Gaussian, and it loses *structurally*: it proposes a vehicle that
cannot exist.

The moving horizon estimator buys its way out with constraints and an optimisation per
sample. Whether you can afford it is what the real-time factor answers, and that depends on
your windows, your solver and your hardware.

Paper: [doi:10.1016/j.conengprac.2023.105747](https://doi.org/10.1016/j.conengprac.2023.105747)  
Code: [github.com/nahueldeniz/nmhec](https://github.com/nahueldeniz/nmhec)